# 05 - Evaluation & Analysis
## PhishScamSense: Real-Time Multimodal Phishing Defense

This notebook covers:
1. Loading trained model checkpoints from notebook 04
2. Building a held-out test set from CIC-Bell-DNS2021
3. 4-class classification metrics: Accuracy, Macro-F1, per-class Precision/Recall
4. Confusion matrix (4×4) visualization
5. Per-class ROC curves (one-vs-rest)
6. XGBoost feature importance analysis
7. Error analysis: which classes get confused with which

In [ ]:
import sys
import os
import pickle
import logging
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import DistilBertModel, DistilBertTokenizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split

from ml.src.data.data_loader import load_cic_bell_dns2021
from ml.src.features.url_features import extract_url_features

sns.set_theme(style="whitegrid")
logging.basicConfig(level=logging.WARNING)

CLASS_NAMES    = ["benign", "phishing", "malware", "spam"]
CLASS_COLORS   = ["#2ecc71", "#e74c3c", "#e67e22", "#9b59b6"]
NUM_CLASSES    = 4
CHECKPOINT_DIR = Path(PROJECT_ROOT) / "ml" / "checkpoints"
DATA_DIR       = Path(PROJECT_ROOT) / "data" / "raw"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 5.1 Re-define Models & Load Checkpoints

In [ ]:
# ── Model classes — attribute names must match ml/src/models/ exactly ────────
class AttentionLayer(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)
    def forward(self, lstm_output):
        weights = torch.softmax(self.attention(lstm_output), dim=1)
        return torch.sum(weights * lstm_output, dim=1)

class NLPBranch(nn.Module):
    def __init__(self, output_dim=128, lstm_hidden=256, lstm_layers=2, dropout=0.3, freeze_bert=True):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        if freeze_bert:
            for p in self.distilbert.parameters(): p.requires_grad = False
        bh = self.distilbert.config.hidden_size
        self.bilstm = nn.LSTM(bh, lstm_hidden, lstm_layers, batch_first=True, bidirectional=True,
                              dropout=dropout if lstm_layers > 1 else 0)
        self.attention = AttentionLayer(lstm_hidden * 2)
        self.fc = nn.Linear(lstm_hidden * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, input_ids, attention_mask):
        x = self.distilbert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        x, _ = self.bilstm(x)
        return self.fc(self.dropout(self.attention(x)))

class MLPBranch(nn.Module):
    def __init__(self, input_dim=23, hidden_dims=None, output_dim=64, dropout=0.3):
        super().__init__()
        if hidden_dims is None: hidden_dims = [128, 64]
        layers = []; prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]; prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.network = nn.Sequential(*layers)
    def forward(self, x): return self.network(x)

class PhishScamSenseFusionModel(nn.Module):
    def __init__(self, num_features=23, nlp_output_dim=128, numerical_output_dim=64, freeze_bert=True):
        super().__init__()
        self.nlp_branch = NLPBranch(output_dim=nlp_output_dim, freeze_bert=freeze_bert)
        self.numerical_branch = MLPBranch(input_dim=num_features, output_dim=numerical_output_dim)
        self.fusion_dim = nlp_output_dim + numerical_output_dim
    def forward(self, input_ids, attention_mask, numerical_features):
        return torch.cat([self.nlp_branch(input_ids, attention_mask),
                          self.numerical_branch(numerical_features)], dim=1)

class URLTokenizer:
    def __init__(self, max_length=128):
        self.tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
        self.max_length = max_length
    def tokenize(self, urls):
        return self.tokenizer(urls, padding=True, truncation=True,
                              max_length=self.max_length, return_tensors="pt")

# ── Load checkpoints ──────────────────────────────────────────────────────────
fusion_model = PhishScamSenseFusionModel(num_features=23)
ckpt_path = CHECKPOINT_DIR / "fusion_model.pt"
if ckpt_path.exists():
    fusion_model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print("Fusion model loaded from checkpoint.")
else:
    print("WARNING: No checkpoint found. Run notebook 04 first.")

xgb_path = CHECKPOINT_DIR / "xgb_classifier.pkl"
if xgb_path.exists():
    with open(xgb_path, "rb") as f:
        xgb_clf = pickle.load(f)
    print("XGBoost model loaded from checkpoint.")
else:
    print("WARNING: No XGBoost checkpoint found. Run notebook 04 first.")

fusion_model.eval()
tokenizer = URLTokenizer()

## 5.2 Build Test Set from CIC-Bell-DNS2021

In [ ]:
import random

TEST_PER_CLASS = 500   # held-out test samples per class
SEED = 99              # different seed from NB04 to avoid overlap

urls_all, labels_all = load_cic_bell_dns2021(DATA_DIR, max_benign=TEST_PER_CLASS * 4, seed=SEED)

rng = random.Random(SEED)
per_class: dict[int, list[str]] = {i: [] for i in range(NUM_CLASSES)}
for url, lbl in zip(urls_all, labels_all):
    per_class[lbl].append(url)

test_urls, test_labels = [], []
for lbl, url_list in per_class.items():
    sample = rng.sample(url_list, min(TEST_PER_CLASS, len(url_list)))
    test_urls.extend(sample)
    test_labels.extend([lbl] * len(sample))
    print(f"  {CLASS_NAMES[lbl]:12s}: {len(sample):,}")

test_urls   = np.array(test_urls,   dtype=object)
test_labels = np.array(test_labels, dtype=np.int32)

# Extract features and embed
num_feat_test = torch.tensor(
    [list(extract_url_features(u).values()) for u in test_urls],
    dtype=torch.float32,
)

CHUNK = 64
all_emb = []
fusion_model.eval()
for i in range(0, len(test_urls), CHUNK):
    batch_urls = test_urls[i:i+CHUNK].tolist()
    enc = tokenizer.tokenize(batch_urls)
    with torch.no_grad():
        emb = fusion_model(enc["input_ids"].to(device),
                           enc["attention_mask"].to(device),
                           num_feat_test[i:i+CHUNK].to(device)).cpu().numpy()
    all_emb.append(emb)

test_emb = np.vstack(all_emb)
y_true   = test_labels
y_pred   = xgb_clf.predict(test_emb).astype(int)
y_proba  = xgb_clf.predict_proba(test_emb)   # shape (n, 4)

print(f"\nTest set embedded: {test_emb.shape}")
print(f"Predictions complete.")

## 5.3 Classification Metrics

In [ ]:
acc      = accuracy_score(y_true, y_pred)
macro_p  = precision_score(y_true, y_pred, average="macro",  zero_division=0)
macro_r  = recall_score(y_true, y_pred,    average="macro",  zero_division=0)
macro_f1 = f1_score(y_true, y_pred,        average="macro",  zero_division=0)
wt_f1    = f1_score(y_true, y_pred,        average="weighted", zero_division=0)

print("=" * 45)
print("  4-CLASS EVALUATION METRICS")
print("=" * 45)
print(f"  {'Accuracy':<20} {acc:.4f}")
print(f"  {'Macro Precision':<20} {macro_p:.4f}")
print(f"  {'Macro Recall':<20} {macro_r:.4f}")
print(f"  {'Macro F1':<20} {macro_f1:.4f}")
print(f"  {'Weighted F1':<20} {wt_f1:.4f}")
print("=" * 45)
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

## 5.4 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 4×4 confusion matrix ─────────────────────────────────────────────────────
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, vmin=0, vmax=1)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix (row-normalised)")

# Raw counts annotation overlay
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        axes[0].text(j + 0.5, i + 0.75, f"n={cm[i,j]}",
                     ha="center", va="center", fontsize=7, color="grey")

# ── Per-class metric bars ─────────────────────────────────────────────────────
per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
axes[1].bar(CLASS_NAMES, per_class_f1, color=CLASS_COLORS, edgecolor="white")
axes[1].set_ylim(0, 1.1)
axes[1].set_title("Per-Class F1 Score")
axes[1].set_ylabel("F1")
for i, v in enumerate(per_class_f1):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nRaw confusion matrix:")
print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_string())

## 5.5 ROC Curve & Precision-Recall Curve

In [ ]:
# Per-class ROC curves (one-vs-rest)
y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))  # (n, 4)

fig, ax = plt.subplots(figsize=(9, 6))
for i, (name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name}  (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("One-vs-Rest ROC Curves — 4 Classes")
ax.legend(loc="lower right")
ax.grid(True)
plt.tight_layout()
plt.show()

## 5.6 XGBoost Feature Importance

In [ ]:
FEATURE_NAMES = [
    "url_length", "hostname_length", "path_length", "num_dots", "num_hyphens",
    "num_underscores", "num_slashes", "num_query_params", "num_fragments",
    "num_digits", "num_special_chars", "url_entropy", "hostname_entropy",
    "has_ip_address", "has_punycode", "has_port", "has_https", "has_at_symbol",
    "has_double_slash_redirect", "subdomain_count", "tld_length",
    "consecutive_consonants_max", "vowel_ratio",
]

# Fused dims: 0-127 = NLP branch, 128-191 = numerical branch
importance  = xgb_clf.feature_importances_
nlp_labels  = [f"nlp_{i}" for i in range(128)]
num_labels  = FEATURE_NAMES          # 23 numerical features
all_labels_imp = nlp_labels + num_labels

top_k = 20
sorted_idx  = np.argsort(importance)[::-1][:top_k]
top_labels  = [all_labels_imp[i] for i in sorted_idx]
top_scores  = importance[sorted_idx]
top_colors  = ["#e74c3c" if l.startswith("nlp") else "#3498db" for l in top_labels]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(top_k), top_scores[::-1], color=top_colors[::-1], edgecolor="white")
ax.set_yticks(range(top_k))
ax.set_yticklabels(top_labels[::-1])
ax.set_xlabel("Importance Score")
ax.set_title(f"Top {top_k} XGBoost Feature Importances\n"
             f"(red = NLP branch dims, blue = numerical features)")
plt.tight_layout()
plt.show()

print(f"\nFused feature vector: {len(importance)} dims  "
      f"(128 NLP + {len(FEATURE_NAMES)} numerical = {128 + len(FEATURE_NAMES)} total)")

## 5.7 Error Analysis: False Positives & False Negatives

In [ ]:
results_df = pd.DataFrame({
    "url":        test_urls,
    "true_label": y_true,
    "pred_label": y_pred,
    "true_class": [CLASS_NAMES[i] for i in y_true],
    "pred_class": [CLASS_NAMES[i] for i in y_pred],
    "confidence": y_proba.max(axis=1).round(4),
})
results_df["correct"] = results_df["true_label"] == results_df["pred_label"]

misclassified = results_df[~results_df["correct"]].copy()
print(f"Total misclassified: {len(misclassified):,} / {len(results_df):,}  "
      f"({len(misclassified)/len(results_df)*100:.1f}%)\n")

# Confusion pairs: which classes get confused most?
confusion_pairs = (
    misclassified
    .groupby(["true_class", "pred_class"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print("Most frequent misclassification pairs:")
print(confusion_pairs.to_string(index=False))

# Sample 3 errors per confusion pair
print("\n--- Misclassified URL samples ---")
for _, row in confusion_pairs.head(6).iterrows():
    subset = misclassified[
        (misclassified["true_class"] == row["true_class"]) &
        (misclassified["pred_class"] == row["pred_class"])
    ].head(2)
    for _, s in subset.iterrows():
        print(f"  [{s['true_class']} → {s['pred_class']}]  conf={s['confidence']:.3f}  {s['url'][:80]}")